In [1]:
from data import *

import numpy as np

In [2]:
# generate data
d1 = 1000
d2 = 100
r = 1
p = 2.0 / d2

M = get_random_matrix(d1, d2, r) / np.sqrt(d1)
observed_M, masks = get_uniformly_random_samples(M, p)

In [3]:
#eps = 0.01

cov_observe_M =  observed_M.T @ observed_M
cov_observe_count = (observed_M == 0).T @ (observed_M == 0)
diag_cov = np.diag( np.diag(cov_observe_M) )

np.count_nonzero(cov_observe_M)

3528

In [4]:
cov_observe_count = (1 * (observed_M != 0)).T @ (1 * (observed_M != 0))
cov_observe_count = cov_observe_count + (cov_observe_count == 0) * 1
T = cov_observe_M / (cov_observe_count/d1)

In [5]:
T_masks = np.nonzero(T)
MTM = M.T @ M

mask_err = T[T_masks] - MTM[T_masks]

print(np.linalg.norm(mask_err) / np.linalg.norm(MTM, 'fro'))

0.5753325893804355


In [6]:
T_p = (1.0 / p) * diag_cov + (1.0 / (p**2)) * (cov_observe_M - diag_cov)

mask_err_Tp = T_p[T_masks] - MTM[T_masks]

print(np.linalg.norm(mask_err_Tp) / np.linalg.norm(MTM, 'fro'))

2.150389448454141


In [21]:
# impute missing values from rank-r SVD corresponding to masks

epochs = 30
#lr = 0.05
X = T
T_masks = 1 * (T != 0)
for i in range(epochs):
    U, D, Vt = np.linalg.svd(X)
    D[r:] = 0
    X_update = U @ np.diag(D) @ Vt

    X = T * T_masks + X_update * (1 - T_masks)

    err = M.T @ M - X
    relative_err = np.linalg.norm(err, 'fro') / np.linalg.norm(M.T @ M, 'fro') 
    print(relative_err)

0.5767393154844228
0.4185153633995495
0.31174979001310843
0.23751020131313072
0.18486336846300952
0.14709507486121667
0.11988659153580096
0.1003393623759336
0.08641997500815086
0.07663856579772466
0.06986841344394659
0.06524972728792428
0.06213562707618593
0.06005291341938436
0.05866608359537483
0.057743659282788275
0.05712917407235821
0.05671831005364673
0.056442085127601634
0.05625506807005575
0.05612737136065066
0.056039314661192066
0.055977907859891554
0.0559345468977729
0.05590350785598936
0.05588096299262005
0.055864337072701296
0.05585188543230821
0.05584241665131119
0.05583510970336668


In [19]:
# run soft impute
epochs = 1
lr = 0.05
X = T
for i in range(epochs):
    U, D, Vt = np.linalg.svd(X)
    D[r:] = 0
    X_update = U @ np.diag(D) @ Vt

    #if np.linalg.norm(X - X_update, 'fro') / np.linalg.norm(X) < eps:
    #    print(i)
    #    break

    X = (1 - lr) * X + lr * X_update

    # print distance
    err = M.T @ M - X
    relative_err = np.linalg.norm(err, 'fro') / np.linalg.norm(M.T @ M)
    print(relative_err)

#print(D)

0.9843367612526361


In [10]:
err = M.T @ M - T
relative_err = np.linalg.norm(err, 'fro') / np.linalg.norm(M.T @ M)
print(relative_err)

0.9850101697937382


In [15]:
cov_observe_count

array([[3, 1, 1, ..., 1, 1, 1],
       [1, 5, 1, ..., 1, 1, 1],
       [1, 1, 5, ..., 1, 1, 1],
       ...,
       [1, 1, 1, ..., 4, 1, 1],
       [1, 1, 1, ..., 1, 7, 1],
       [1, 1, 1, ..., 1, 1, 7]])

In [16]:
T

array([[4.41850813, 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 4.40887384, 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 3.9159747 , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 3.85412764, 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 3.83542448,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        4.14661854]])